# 02 — Single-Factor Strategy Exploration

Systematic grid search across 13 firm characteristics, varying:
- Percentile cutoffs (30/70, 10/90, 5/95, 1/99)
- Weight schemes (equal, value, char_rank_weighted, char_minmax_weighted)
- Max weight per leg (uncapped, 5%, 10%, 20%)

Total: 832 strategy configurations → ranked by Sharpe ratio.

## Characteristics tested

| Theme | Variable | Expected Sign |
|-------|----------|---------------|
| Size | `market_equity` | negative |
| Value | `be_me` | positive |
| Profitability | `op_at`, `ni_be` | positive |
| Quality | `qmj` | positive |
| Investment | `at_gr1` | negative |
| Accruals | `oaccruals_at` | negative |
| Debt issuance | `dbnetis_at` | negative |
| Leverage | `debt_at` | negative |
| Profit growth | `niq_at_chg1` | positive |
| Momentum | `ret_6_1` | positive |
| Short-term reversal | `ret_1_0` | negative |
| Low risk | `beta_60m` | negative |

In [1]:
# ── Environment setup (run once) ──────────────────────────────────────
import sys, os, subprocess

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_PATH = "/content/drive/MyDrive/quant-trading-experiments"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_PATH}[dev]"])

    JKP_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"
else:
    JKP_CSV_PATH = "/Users/abhin/Library/CloudStorage/GoogleDrive-abhinavkeshri1999@gmail.com/My Drive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"

print(f"Data path: {JKP_CSV_PATH}")
print(f"Exists: {os.path.exists(JKP_CSV_PATH)}")

try:
    import quant_trading
    print(f"quant_trading v{quant_trading.__version__} imported OK")
except ImportError:
    print("quant_trading not importable yet — restart kernel and run this cell again")

Data path: /Users/abhin/Library/CloudStorage/GoogleDrive-abhinavkeshri1999@gmail.com/My Drive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv
Exists: True
quant_trading v0.1.0 imported OK


In [2]:
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from quant_trading.data import (
    load_jkp_csv, winsorize_returns, CHAR_DIRECTIONS,
    TARGET_COL, NUMERIC_FEATURES,
)
from quant_trading.signals import generate_positions
from quant_trading.portfolio import calculate_portfolio_returns
from quant_trading.evaluation import perf_stats_annualized

/Users/abhin/pyenv_jupyter/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# JKP_CSV_PATH is set in the setup cell above
df_jkp = load_jkp_csv(JKP_CSV_PATH)
df_jkp = df_jkp.drop_duplicates(subset=["month_date", "id"])
df_jkp = df_jkp.dropna(subset=["id", "eom", "ret_exc_lead1m"])
df_jkp = df_jkp.drop_duplicates(subset=["eom", "ret_exc_lead1m"])

winsorization_value = 0.05 / 100
df_jkp = winsorize_returns(
    df_jkp,
    lower=winsorization_value,
    upper=1 - winsorization_value,
)

KEEP_COLS = ["id", "month_date", TARGET_COL] + NUMERIC_FEATURES_ML + CATEGORICAL_FEATURES
KEEP_COLS = list(dict.fromkeys(KEEP_COLS))

missing = [c for c in KEEP_COLS if c not in df_jkp.columns]
if missing:
    print("Missing columns:", missing)

df_jkp = df_jkp[[c for c in KEEP_COLS if c in df_jkp.columns]].copy()

print(f"Loaded {len(df_jkp):,} rows")
print(f"Minimum return: {df_jkp[TARGET_COL].min()}")
print(f"Maximum return: {df_jkp[TARGET_COL].max()}")


/Users/abhin/pyenv_jupyter/.venv/lib/python3.14/site-packages/quant_trading/data.py:106: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, parse_dates=date_cols)


Loaded 5,027,542 rows
Minimum return: -1.001598
Maximum return: 187.33493300007916


In [10]:
# Quick correlation check between features and target
df_core = df_jkp.dropna(subset=[TARGET_COL] + NUMERIC_FEATURES).copy()
corrs = df_core[NUMERIC_FEATURES].corrwith(df_core[TARGET_COL])
print("Feature correlations with target return:")
print(corrs.sort_values(key=abs, ascending=False))

Feature correlations with target return:
be_me            0.030117
ret_1_0         -0.011711
op_at            0.007876
oaccruals_at    -0.004547
at_gr1          -0.004168
market_equity   -0.004056
beta_60m         0.003096
ret_6_1          0.002721
dbnetis_at      -0.002542
ni_be           -0.001862
qmj              0.001543
debt_at         -0.001229
niq_at_chg1     -0.001009
dtype: float64


## Strategy Grid Search

In [ ]:
chars = {
    "market_equity": {"long_high": False},
    "be_me":         {"long_high": True},
    "op_at":         {"long_high": True},
    "oaccruals_at":  {"long_high": False},
    "dbnetis_at":    {"long_high": False},
    "at_gr1":        {"long_high": False},
    "debt_at":       {"long_high": False},
    "niq_at_chg1":   {"long_high": True},
    "ni_be":         {"long_high": True},
    "qmj":           {"long_high": True},
    "ret_1_0":       {"long_high": False},
    "ret_6_1":       {"long_high": True},
    "beta_60m":      {"long_high": False},
}

pct_pairs = [(0.3, 0.7), (0.1, 0.9), (0.05, 0.95), (0.01, 0.99)]
max_weight_per_leg = [1, 0.05, 0.1, 0.2]
rebal_periods = [1]
weight_schemes = ["value", "equal", "char_rank_weighted", "char_minmax_weighted"]

# Build all strategy combinations
strategy_settings = {}

for char_name, meta in chars.items():
    for (lower_pct, upper_pct), rebal_period, weight_scheme in itertools.product(
        pct_pairs, rebal_periods, weight_schemes
    ):
        # only char-based schemes use max_weight
        if weight_scheme in ["equal", "value"]:
            weight_grid = [None]
        else:
            weight_grid = max_weight_per_leg

        for max_weight in weight_grid:
            strategy_name = (
                f"{char_name}"
                f"_L{int(lower_pct * 100)}"
                f"_U{int(upper_pct * 100)}"
                f"_R{rebal_period}"
                f"_{weight_scheme}"
                f"{'' if max_weight is None else f'_W{max_weight}'}"
            )

            strategy_settings[strategy_name] = {
                "char_name": char_name,
                "lower_pct": lower_pct,
                "upper_pct": upper_pct,
                "long_high": meta["long_high"],
                "rebal_period": rebal_period,
                "weight_scheme": weight_scheme,
                "max_weight": max_weight,
            }

print(f"Total strategy configurations: {len(strategy_settings)}")

Total strategy configurations: 780


In [50]:
freq = 12
summary_rows = []
cumret_dict = {}
ret_dict = {}

for strategy_name, params in tqdm(strategy_settings.items(), desc="Evaluating strategies"):
    char_name = params["char_name"]
    print(params)
    df = df_jkp.dropna(subset=[char_name, "market_equity", "ret_exc_lead1m_w"]).copy()

    char_positions = generate_positions(
        df,
        char_name=char_name,
        lower_pct=params["lower_pct"],
        upper_pct=params["upper_pct"],
        long_high=params["long_high"],
        rebal_period=params["rebal_period"],
    )

    char_returns = calculate_portfolio_returns(
        ret_col="ret_exc_lead1m_w",
        positions=char_positions,
        data=df,
        weight_scheme=params["weight_scheme"],
        weight_col="market_equity",
        char_col=char_name,
        long_high=params["long_high"],
        max_weight_per_leg=params["max_weight"],
        drop_micro_caps=True
    )

    rets = char_returns["Spread"].dropna()
    if rets.empty:
        continue

    ann_mean = rets.mean() * freq
    ann_vol = rets.std(ddof=1) * np.sqrt(freq)
    sharpe = ann_mean / ann_vol if ann_vol != 0 else np.nan
    cum_ret = (1 + rets).cumprod()
    max_drawdown = (cum_ret / cum_ret.cummax() - 1).min()

    summary_rows.append({
        "Strategy": strategy_name,
        "Characteristic": char_name,
        "Long High": params["long_high"],
        "Lower Pct": params["lower_pct"],
        "Upper Pct": params["upper_pct"],
        "Rebal Period": params["rebal_period"],
        "Weight Scheme": params["weight_scheme"],
        "Max weight per leg": params["max_weight"],
        "Months": len(rets),
        "Annualized Mean": ann_mean,
        "Annualized Vol": ann_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_drawdown,
        "Hit Rate": (rets > 0).mean(),
        "Best Month": rets.max(),
        "Worst Month": rets.min(),
    })
    ret_dict[strategy_name] = rets
    cumret_dict[strategy_name] = (1 + rets).cumprod() - 1

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["Sharpe", "Annualized Mean"], ascending=[False, False]).reset_index(drop=True)

Evaluating strategies:   0%|          | 0/5 [00:00<?, ?it/s]

{'char_name': 'ret_1_0', 'lower_pct': 0.1, 'upper_pct': 0.9, 'long_high': False, 'rebal_period': 1, 'weight_scheme': 'char_minmax_weighted', 'max_weight': 1}


Evaluating strategies:  20%|██        | 1/5 [00:06<00:26,  6.52s/it]

{'char_name': 'ret_1_0', 'lower_pct': 0.1, 'upper_pct': 0.9, 'long_high': False, 'rebal_period': 1, 'weight_scheme': 'char_minmax_weighted', 'max_weight': 0.2}


Evaluating strategies:  40%|████      | 2/5 [00:24<00:40, 13.46s/it]

...

Evaluating strategies:  60%|██████    | 3/5 [00:31<00:20, 10.46s/it]

{'char_name': 'ret_1_0', 'lower_pct': 0.1, 'upper_pct': 0.9, 'long_high': False, 'rebal_period': 1, 'weight_scheme': 'char_minmax_weighted', 'max_weight': 0.05}


Evaluating strategies:  80%|████████  | 4/5 [00:38<00:08,  8.87s/it]

{'char_name': 'ret_1_0', 'lower_pct': 0.1, 'upper_pct': 0.9, 'long_high': False, 'rebal_period': 1, 'weight_scheme': 'char_minmax_weighted', 'max_weight': 0.01}


Evaluating strategies: 100%|██████████| 5/5 [00:44<00:00,  8.85s/it]


In [51]:
# Top 10 strategies by Sharpe ratio
top_n = 10
display(
    summary_df.head(top_n).style.format({
        "Lower Pct": "{:.0%}",
        "Upper Pct": "{:.0%}",
        "Max weight per leg": "{:.0%}",
        "Annualized Mean": "{:.2%}",
        "Annualized Vol": "{:.2%}",
        "Sharpe": "{:.2f}",
        "Max Drawdown": "{:.2%}",
        "Hit Rate": "{:.2%}",
        "Best Month": "{:.2%}",
        "Worst Month": "{:.2%}",
    })
)

,Strategy,Characteristic,Long High,Lower Pct,Upper Pct,Rebal Period,Weight Scheme,Max weight per leg,Months,Annualized Mean,Annualized Vol,Sharpe,Max Drawdown,Hit Rate,Best Month,Worst Month
0,ret_1_0_L10_U90_R1_char_minmax_weighted_W0.01,ret_1_0,False,10%,90%,1,char_minmax_weighted,1%,659,71.73%,33.94%,2.11,-51.71%,81.94%,92.86%,-42.27%
1,ret_1_0_L10_U90_R1_char_minmax_weighted_W0.05,ret_1_0,False,10%,90%,1,char_minmax_weighted,5%,659,82.63%,41.45%,1.99,-76.78%,81.49%,100.91%,-63.35%
2,ret_1_0_L10_U90_R1_char_minmax_weighted_W0.1,ret_1_0,False,10%,90%,1,char_minmax_weighted,10%,659,87.81%,45.81%,1.92,-77.87%,80.88%,102.02%,-56.20%
3,ret_1_0_L10_U90_R1_char_minmax_weighted_W0.2,ret_1_0,False,10%,90%,1,char_minmax_weighted,20%,659,94.17%,51.81%,1.82,-94.32%,80.88%,106.19%,-92.62%
4,ret_1_0_L10_U90_R1_char_minmax_weighted_W1,ret_1_0,False,10%,90%,1,char_minmax_weighted,100%,659,102.49%,63.40%,1.62,-21769267.13%,80.88%,112.95%,-180.41%


## Cumulative Returns — Top Strategies

In [37]:
import plotly.graph_objects as go

top_strategies = summary_df.head(top_n)["Strategy"].tolist()
cumret_df = pd.DataFrame(cumret_dict)
cols_to_plot = [c for c in top_strategies if c in cumret_df.columns]

fig = go.Figure()
for i, col in enumerate(cols_to_plot):
    fig.add_trace(go.Scatter(
        x=cumret_df.index, y=cumret_df[col],
        mode="lines", name=col, line=dict(width=2),
        visible=True if i < 5 else "legendonly",
    ))
fig.update_layout(
    title=f"Cumulative Returns — Top {len(cols_to_plot)} Strategies",
    xaxis_title="Date", yaxis_title="Cumulative Return",
    template="plotly_white", hovermode="x unified",
)
fig.show()

In [ ]:
# Save results for downstream analysis
summary_df.to_csv("../data/factor_strategy_summary.csv", index=False)
pd.DataFrame(ret_dict).to_csv("../data/factor_strategy_monthly_returns.csv", index=True)
print("Saved summary and returns to data/")